In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from datasets import Dataset

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

import torch



In [2]:
df = pd.read_csv(
    r"D:\contract-intelligence-ai\data\processed\final_training_dataset.csv"
)

print(df.shape)

df.head()

(1654, 4)


,contract_id,label,text,category
0,1.0,Enrollment in the Affiliate Program; Restricte...,1. Enrollment in the Affiliate Program; Restri...,General
1,1.0,Affiliate Responsibilities:,2. Affiliate Responsibilities:\n• Affiliate ca...,General
2,1.0,Referral Fee,3. Referral Fee\nFor each Approved Account (as...,Payment
3,1.0,Approved Account,4. Approved Account\nFor purposes of determini...,General
4,1.0,Links,6. Links\nAffiliate agrees to place Chase’s li...,General


In [3]:
df = df[["text", "category"]]

df = df.dropna()

print(df.shape)

(1654, 2)


In [4]:
label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(
    df["category"]
)

print(label_encoder.classes_)

['Confidentiality' 'Employment' 'General' 'Intellectual_Property' 'Legal'
 'Liability' 'Payment' 'Performance' 'Property' 'Termination']


In [5]:
import pickle

with open(
    "label_encoder.pkl",
    "wb"
) as f:

    pickle.dump(
        label_encoder,
        f
    )

In [6]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print(train_df.shape)
print(test_df.shape)

(1323, 3)
(331, 3)


In [7]:
train_dataset = Dataset.from_pandas(
    train_df
)

test_dataset = Dataset.from_pandas(
    test_df
)

In [8]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

In [9]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [10]:
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/1323 [00:00<?, ? examples/s]

Map:   0%|          | 0/331 [00:00<?, ? examples/s]

In [11]:
train_dataset = train_dataset.rename_column(
    "label",
    "labels"
)

test_dataset = test_dataset.rename_column(
    "label",
    "labels"
)

train_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

test_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

In [12]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(
        label_encoder.classes_
    )
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=3,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    save_strategy="no",

    report_to="none",

    logging_steps=10
)

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [15]:
trainer.train()

c:\Users\SHREEJA\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


KeyboardInterrupt: 

In [16]:
import pandas as pd

df = pd.read_csv(
    r"D:\contract-intelligence-ai\data\processed\final_training_dataset.csv"
)

print(df.shape)

print(df["category"].value_counts())

(1654, 4)
category
General                  947
Legal                    259
Payment                  102
Termination               89
Liability                 76
Confidentiality           53
Intellectual_Property     52
Property                  34
Performance               26
Employment                16
Name: count, dtype: int64


In [17]:
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [18]:
df = pd.read_csv(
    r"D:\contract-intelligence-ai\data\processed\final_training_dataset.csv"
)

df = df[["text", "category"]]

df = df.dropna()

print(df.shape)

(1654, 2)


In [19]:
label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(
    df["category"]
)

print(label_encoder.classes_)

['Confidentiality' 'Employment' 'General' 'Intellectual_Property' 'Legal'
 'Liability' 'Payment' 'Performance' 'Property' 'Termination']


In [20]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print(train_df.shape)
print(test_df.shape)

(1323, 3)
(331, 3)


In [21]:
print(torch.cuda.is_available())

False


In [22]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df
)

test_dataset = Dataset.from_pandas(
    test_df
)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'category', 'label', '__index_level_0__'],
    num_rows: 1323
})
Dataset({
    features: ['text', 'category', 'label', '__index_level_0__'],
    num_rows: 331
})


In [23]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

In [24]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [25]:
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/1323 [00:00<?, ? examples/s]

Map:   0%|          | 0/331 [00:00<?, ? examples/s]

In [26]:
train_dataset = train_dataset.rename_column(
    "label",
    "labels"
)

test_dataset = test_dataset.rename_column(
    "label",
    "labels"
)

train_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

test_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

print("Dataset Ready")

Dataset Ready


In [27]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(
        label_encoder.classes_
    )
)

print("Model Loaded Successfully")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model Loaded Successfully


In [28]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=2,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    save_strategy="no",

    report_to="none",

    logging_steps=25
)

print("Training Arguments Ready")

Training Arguments Ready


In [29]:
from transformers import Trainer

trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset
)

print("Trainer Ready")

Trainer Ready


In [30]:
print(len(train_dataset))
print(len(test_dataset))

1323
331


In [31]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results"
)

print("Works")

Works


In [32]:
trainer.train()

AttributeError: `AcceleratorState` object has no attribute `distributed_type`. This happens if `AcceleratorState._reset_state()` was called and an `Accelerator` or `PartialState` was not reinitialized.

In [33]:
import sys
print(sys.executable)

c:\Users\SHREEJA\anaconda3\python.exe
